#### Dataset EDA

Exploration for the AlphaEarth and NLCD wetland sample data. Two parts: the
raw joined table, which does not depend on any generated dataset, and one
generated training pool, chosen below by name. Supersedes the EDA and
"Training pool composition by fold" sections of `gradient_boosting.ipynb`,
which mixed both together with the model training code.


In [ ]:
import os
import sys

sys.path.append(os.path.abspath("scripts"))

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

import gb_common

pd.set_option("display.max_columns", None)


## Part 1: Raw joined table

Check the joined AlphaEarth and NLCD samples on their own terms, before any
dataset gets built from them: label balance, embedding value ranges, whether
the granule seam problem seen on some coastal tiles during the AlphaEarth
export shows up here, where samples fall geographically, and whether
`dist_to_developed_2019_m` behaves the way the labeling scheme implies.
Queries run against the parquet directory in place with DuckDB instead of
loading all 59M rows into pandas. The joined tiles are 9GB on disk, about
46GB as a dense float32 frame, well past what fits in memory here.


In [ ]:
con = gb_common.connect_raw()
con.execute("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT tile_id) AS n_tiles FROM samples").df()


### Label balance

This number decides how any dataset built from this table gets split and
weighted for training. Expect "Remained wetland" to dominate heavily, since
conversion is a rare event over a 5 year window.


In [ ]:
label_counts = con.execute("""
    SELECT label, label_name, COUNT(*) AS n
    FROM samples
    GROUP BY label, label_name
    ORDER BY label
""").df()
label_counts["frac"] = label_counts["n"] / label_counts["n"].sum()
label_counts


### Embedding value range

`export_alphaearth_30m.py` decodes the raw uint8 bands back to `[-1, 1]`
using `OFFSET` and `SCALE_FACTOR`. If the decode or the join went wrong
somewhere, this is the first place it would show: either the range is off,
or a NODATA fill leaked through (the assemble step is supposed to drop
those, not leave them in).


In [ ]:
band_bounds = con.execute(f"""
    SELECT MIN(v) AS min_val, MAX(v) AS max_val, COUNT(*) AS n_vals
    FROM samples
    UNPIVOT (v FOR band IN ({", ".join(gb_common.BAND_COLS)}))
""").df()
band_bounds


### Coastal tile screen

The AlphaEarth export run turned up nondeterminism in the seams between
granules on some coastal tiles. This is a coarse check for whether that
shows up here before trusting the pooled data. Row counts per tile catch a
tile that is missing chunks of coverage. Stats per tile on a single band
(A00_2019, an arbitrary choice) catch a tile whose embedding distribution is
obviously off from its neighbors. This is a screen, not a diagnosis:
anything that stands out here is worth checking against the raw tif before
deciding whether to drop or reprocess that tile.


In [ ]:
tile_stats = con.execute("""
    SELECT tile_id,
           COUNT(*) AS n,
           AVG(A00_2019) AS a00_mean,
           STDDEV(A00_2019) AS a00_std,
           MIN(A00_2019) AS a00_min,
           MAX(A00_2019) AS a00_max
    FROM samples
    GROUP BY tile_id
    ORDER BY tile_id
""").df()

# flag tiles whose A00_2019 mean sits far from the mean across tiles: a cheap
# proxy for "this tile's embeddings look systematically different"
overall_mean, overall_std = tile_stats["a00_mean"].mean(), tile_stats["a00_mean"].std()
tile_stats["z"] = (tile_stats["a00_mean"] - overall_mean) / overall_std
tile_stats.reindex(tile_stats["z"].abs().sort_values(ascending=False).index).head(10)


### Spatial distribution

Pull a random sample (small enough to plot, not meant to be representative
of class balance) and check that coverage looks like Florida's wetlands, and
that converted pixels are not clustered somewhere that smells like a
processing artifact rather than real land cover change.


In [ ]:
spatial_sample = con.execute("""
    SELECT x, y, label_name
    FROM samples
    USING SAMPLE 200000 ROWS
""").df()

with plt.rc_context(gb_common.SLIDE_RC):
    fig, ax = plt.subplots(figsize=(6, 8))
    for name, grp in spatial_sample.groupby("label_name"):
        # converted classes plotted last, on top, so they aren't buried under
        # the much larger "remained wetland" cloud
        z = 2 if name == "Remained wetland" else 3
        ax.scatter(grp["x"], grp["y"], s=1, alpha=0.4, color=gb_common.LABEL_COLORS[name], label=name, zorder=z)
    ax.set_aspect("equal")
    ax.set_xlabel("Easting (m, NLCD Albers)")
    ax.set_ylabel("Northing (m, NLCD Albers)")
    ax.set_title(f"{len(spatial_sample):,} random sample locations by label")
    ax.legend(markerscale=15, loc="upper left")
    plt.show()


### `dist_to_developed_2019_m` versus label

Distance to the nearest 2019 developed pixel is one of the strongest priors
for conversion risk: proximity to existing development. Pixels that
converted between 2019 and 2024 should sit closer to 2019 development on
average than pixels that stayed wetland. If they do not, that is worth
understanding before this column gets treated as a trustworthy feature.


In [ ]:
dist_by_label = con.execute(f"""
    SELECT label_name,
           COUNT(*) AS n,
           AVG({gb_common.DIST_COL}) AS mean_dist_m,
           MEDIAN({gb_common.DIST_COL}) AS median_dist_m
    FROM samples
    GROUP BY label_name
    ORDER BY mean_dist_m
""").df()
dist_by_label


In [ ]:
# label=0 outnumbers label=1 by about 350 to 1, so one shared linear axis
# would flatten the conversion classes to nothing: bin the distance itself in
# log space instead, since it spans 30m (one AlphaEarth pixel) to over 100km
# and is heavily right skewed
LOG_BIN_WIDTH = 0.05  # 20 bins per decade
DIST_TICKS = [30, 100, 300, 1000, 3000, 10000, 30000, 100000]

dist_hist = con.execute(f"""
    SELECT label_name,
           FLOOR(LOG10({gb_common.DIST_COL}) / {LOG_BIN_WIDTH}) AS bin_idx,
           COUNT(*) AS n
    FROM samples
    GROUP BY label_name, bin_idx
""").df()
dist_hist["left_m"] = 10 ** (dist_hist["bin_idx"] * LOG_BIN_WIDTH)
dist_hist["right_m"] = 10 ** ((dist_hist["bin_idx"] + 1) * LOG_BIN_WIDTH)


In [ ]:
with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=False)
    for ax, label_name in zip(axes, gb_common.LABEL_COLORS):
        grp = dist_hist[dist_hist["label_name"] == label_name].sort_values("left_m")
        stats = dist_by_label.set_index("label_name").loc[label_name]

        ax.bar(grp["left_m"], grp["n"], width=grp["right_m"] - grp["left_m"],
               align="edge", color=gb_common.LABEL_COLORS[label_name], edgecolor="none")
        ax.set_xscale("log")
        ax.set_xticks(DIST_TICKS)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.set_xlabel("Distance to development (m)")
        ax.set_title(label_name, fontsize=12)
        ax.tick_params(axis="x", rotation=30)

        ax.axvline(stats["mean_dist_m"], color="black", linestyle="--", linewidth=1.3)
        ax.axvline(stats["median_dist_m"], color="black", linestyle=":", linewidth=1.3)
        ymax = ax.get_ylim()[1]
        ax.text(stats["mean_dist_m"] * 1.1, ymax * 0.92, f"mean {stats['mean_dist_m']:,.0f} m", va="top", fontsize=9)
        ax.text(stats["median_dist_m"] * 1.1, ymax * 0.68, f"median {stats['median_dist_m']:,.0f} m", va="top", fontsize=9)
    axes[0].set_ylabel("Frequency")
    fig.suptitle("Distance to nearest 2019 developed pixel, by label", y=1.03, fontsize=14)
    fig.tight_layout()
    plt.show()


## Part 2: Training pool

Everything below is EDA on one generated dataset, the fold assignment and
weighted training pool `scripts/build_train_pool.py` writes to
`data/processed/datasets/<DATASET_NAME>/`. Set `DATASET_NAME` to whichever
dataset was just built (or already exists) to inspect it. Nothing here
writes anything: this is a read only look at what a dataset build produced.


In [ ]:
DATASET_NAME = "baseline-v1"

con_pool, block_folds = gb_common.connect_samples(DATASET_NAME)
train_pool = gb_common.load_train_pool(DATASET_NAME)
dataset_config = gb_common.load_dataset_config(DATASET_NAME)
dataset_config


### Assigning blocks to folds

`block_id` (about 10km cells, baked in by `join_alphaearth_samples.py`'s
`add_block_id`) exists so spatially correlated neighboring pixels do not
straddle a train and test boundary. `scripts/build_train_pool.py folds`
balances three things at once instead of just positives: block count, the
label=1 and label=2 counts, and total rows.


In [ ]:
block_folds.groupby("fold").agg(
    blocks=("block_id", "size"), rows=("n", "sum"), n1=("n1", "sum"), n2=("n2", "sum")
)


### Pool integrity

A pool drawn against a superseded `block_folds.parquet` would still load
fine and silently train on the wrong split, so this checks the parts that
would differ rather than trusting the file blindly.


In [ ]:
pool_blocks = train_pool[["block_id", "fold"]].drop_duplicates()
assert pool_blocks.merge(block_folds[["block_id", "fold"]], on=["block_id", "fold"]).shape[0] == len(pool_blocks), \
    "pool fold assignment disagrees with block_folds.parquet, rerun build_train_pool.py"
assert (train_pool.loc[train_pool["label"] == 1, "sample_weight"]
        == dataset_config["pos_weight_mult"] / dataset_config["pos_row_dup"]).all()

len(train_pool), train_pool[gb_common.TARGET_COL].mean()


### Per fold training set size and weight share

Two numbers describe this pool and they disagree by roughly four times. By
row count it is around 10% positive. By weight, which is what LightGBM
actually optimizes since it accumulates weighted gradients and hessians, it
is closer to 3%. The row figure is the misleading one.


In [ ]:
pool_rows_by_fold = train_pool.groupby("fold").size().rename("pool_rows")
fold_sizes = pd.DataFrame({
    "pool_rows_held_out": pool_rows_by_fold,
    "training_rows": len(train_pool) - pool_rows_by_fold,
})
display(fold_sizes)

flagged = train_pool.assign(
    is_pos=(train_pool["label"] == 1).astype(float),
    pos_w=train_pool["sample_weight"].where(train_pool["label"] == 1, 0.0),
)
share = flagged.groupby("fold").agg(
    rows_pct=("is_pos", "mean"),
    pos_weight=("pos_w", "sum"),
    total_weight=("sample_weight", "sum"),
    mean_weight=("sample_weight", "mean"),
)
share["rows_pct"] *= 100
share["weight_pct"] = 100 * share["pos_weight"] / share["total_weight"]
share[["rows_pct", "weight_pct", "mean_weight"]]


### Fold distance profile, natural versus pool

The ECDF is read straight off quantiles rather than by pulling 59M distances
into pandas: 199 evenly spaced quantiles per fold trace the same curve. The
left panel is each fold as it naturally occurs in Florida; the right panel
is after the distance weighted draw thinned label=0 rows down to the
training pool.


In [ ]:
ECDF_PROBS = [i / 200 for i in range(1, 200)]
_probs_sql = "[" + ", ".join(f"{p:.5f}" for p in ECDF_PROBS) + "]"

natural_ecdf = con_pool.execute(f"""
    SELECT fold, QUANTILE_CONT({gb_common.DIST_COL}, {_probs_sql}) AS xs
    FROM samples_fold WHERE label = 0
    GROUP BY fold ORDER BY fold
""").df()

pool_ecdf = (
    train_pool[train_pool["label"] == 0]
    .groupby("fold")[gb_common.DIST_COL]
    .quantile(ECDF_PROBS)
    .unstack(level=-1)
)

with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), sharey=True)
    for k in range(gb_common.K_FOLDS):
        axes[0].plot(natural_ecdf.loc[natural_ecdf["fold"] == k, "xs"].iloc[0], ECDF_PROBS,
                     color=gb_common.FOLD_COLORS[k], lw=2, label=f"fold {k}")
        axes[1].plot(pool_ecdf.loc[k].to_numpy(), ECDF_PROBS,
                     color=gb_common.FOLD_COLORS[k], lw=2, label=f"fold {k}")

    axes[0].set_title("Full fold, as it occurs in Florida")
    axes[1].set_title("After the distance weighted draw")
    axes[0].set_ylabel("Share of wetland pixels at or below this distance")
    for ax in axes:
        ax.set_xscale("log")
        ax.set_xlim(25, 120_000)
        ax.set_xlabel("Distance to 2019 development (m, log scale)")
        ax.set_xticks([30, 100, 300, 1000, 3000, 10_000, 30_000, 100_000])
        ax.set_xticklabels(["30", "100", "300", "1k", "3k", "10k", "30k", "100k"])
        ax.grid(axis="y", alpha=0.25, lw=0.6)
        ax.set_axisbelow(True)
    axes[0].set_ylim(0, 1)
    axes[0].legend(loc="lower right", frameon=False, ncol=2)
    plt.show()


### Pool composition by fold

The block balancing above evened out raw block counts and positive counts
before any sampling happened. The distance weighted draw then thinned
label=0 rows nonuniformly, which could in principle throw that balance off
again if one fold's blocks happen to sit at systematically different
distances from development than another's. This checks the artifact that
actually gets trained on.


In [ ]:
pool_counts = (
    train_pool.groupby(["fold", "label"]).size().unstack(fill_value=0)
    .rename(columns={0: "Remained wetland", 1: "Converted to developed",
                     2: "Converted to other, not developed"})
)
CLASS_COLORS = {"Remained wetland": gb_common.LABEL_COLORS["Remained wetland"],
                "Converted to other, not developed": gb_common.LABEL_COLORS["Converted to other (non-developed)"],
                "Converted to developed": gb_common.LABEL_COLORS["Converted to developed"]}

with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

    width, folds = 0.26, np.arange(gb_common.K_FOLDS)
    for i, name in enumerate(CLASS_COLORS):
        axes[0].bar(folds + (i - 1) * width, pool_counts[name], width,
                    color=CLASS_COLORS[name], label=name)
    axes[0].set_yscale("log")
    axes[0].set_xticks(folds)
    axes[0].set_xlabel("Cross validation fold")
    axes[0].set_ylabel("Rows in the training pool (log scale)")
    axes[0].set_title("Pool composition, by row count")
    axes[0].legend(loc="upper center", frameon=False, fontsize=11, ncol=1)
    axes[0].grid(axis="y", alpha=0.25, lw=0.6)
    axes[0].set_axisbelow(True)

    axes[1].bar(folds - width / 2, share["rows_pct"], width, color="#999999", label="Share of rows")
    axes[1].bar(folds + width / 2, share["weight_pct"], width, color="#D55E00",
                label="Share of weight, what the loss sees")
    for k in folds:
        axes[1].text(k - width / 2, share["rows_pct"].iloc[k] + 0.3, f"{share['rows_pct'].iloc[k]:.1f}%",
                     ha="center", fontsize=11)
        axes[1].text(k + width / 2, share["weight_pct"].iloc[k] + 0.3, f"{share['weight_pct'].iloc[k]:.1f}%",
                     ha="center", fontsize=11)
    axes[1].set_xticks(folds)
    axes[1].set_xlabel("Cross validation fold")
    axes[1].set_ylabel("Converted to developed (percent)")
    axes[1].set_title("The same pool, measured two ways")
    axes[1].legend(loc="upper center", frameon=False, fontsize=11)
    axes[1].grid(axis="y", alpha=0.25, lw=0.6)
    axes[1].set_axisbelow(True)
    plt.show()


In [ ]:
fold_label_counts = train_pool.groupby(["fold", gb_common.TARGET_COL]).size().rename("n").reset_index()
categories = ["Did not convert\nto developed", "Converted\nto developed"]

with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, gb_common.K_FOLDS, figsize=(4 * gb_common.K_FOLDS, 4.5), sharey=True)
    for fold_id, ax in enumerate(axes):
        grp = fold_label_counts[fold_label_counts["fold"] == fold_id].set_index(gb_common.TARGET_COL)["n"]
        counts = [grp.get(0, 0), grp.get(1, 0)]
        bars = ax.bar(categories, counts, color=gb_common.FOLD_COLORS[fold_id])
        ax.set_title(f"Fold {fold_id}")
        ax.tick_params(labelsize=10)
        for bar, c in zip(bars, counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{c:,}",
                    ha="center", va="bottom", fontsize=9)
    axes[0].set_ylabel("Rows in training pool")
    fig.tight_layout()
    plt.show()


In [ ]:
bin_idx = np.floor(np.log10(train_pool[gb_common.DIST_COL]) / LOG_BIN_WIDTH)
fold_dist_hist = (
    pd.DataFrame({"fold": train_pool["fold"], "bin_idx": bin_idx})
    .groupby(["fold", "bin_idx"]).size().rename("n").reset_index()
)
fold_dist_hist["left_m"] = 10 ** (fold_dist_hist["bin_idx"] * LOG_BIN_WIDTH)
fold_dist_hist["right_m"] = 10 ** ((fold_dist_hist["bin_idx"] + 1) * LOG_BIN_WIDTH)

with plt.rc_context(gb_common.SLIDE_RC):
    fig, axes = plt.subplots(1, gb_common.K_FOLDS, figsize=(4.5 * gb_common.K_FOLDS, 4.5), sharey=True)
    for fold_id, ax in enumerate(axes):
        grp = fold_dist_hist[fold_dist_hist["fold"] == fold_id].sort_values("left_m")
        dist_vals = train_pool.loc[train_pool["fold"] == fold_id, gb_common.DIST_COL]
        mean_d, median_d = dist_vals.mean(), dist_vals.median()

        ax.bar(grp["left_m"], grp["n"], width=grp["right_m"] - grp["left_m"],
               align="edge", color=gb_common.FOLD_COLORS[fold_id], edgecolor="none")
        ax.set_xscale("log")
        ax.set_xticks(DIST_TICKS)
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.set_xlabel("Distance to development (m)", fontsize=10)
        ax.set_title(f"Fold {fold_id}")
        ax.axvline(mean_d, color="black", linestyle="--", linewidth=1.1)
        ax.axvline(median_d, color="black", linestyle=":", linewidth=1.1)
        ax.tick_params(labelsize=9, axis="x", rotation=30)
    axes[0].set_ylabel("Rows in training pool")
    fig.tight_layout()
    plt.show()
